In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# 1. Load dataset
train_data = tf.keras.utils.image_dataset_from_directory(
    'Object Detection/caltech-101',
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(128,128),
    batch_size=64
)

test_data = tf.keras.utils.image_dataset_from_directory(
    'Object Detection/caltech-101',
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(128,128),
    batch_size=64
)

num_classes = len(train_data.class_names)

# 2. Load offline VGG16 weights
base_model = VGG16(
    weights='Object Detection/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5',
    include_top=False,
    input_shape=(128,128,3)
)

base_model.trainable = False

# 3. Build model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128,128,3)),
    tf.keras.layers.Lambda(preprocess_input),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# 4. Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train model
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=3
)

# 6. Evaluate model
loss, accuracy = model.evaluate(test_data)
print("Test Accuracy:", accuracy)

# 7. Plot accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.legend(['Training', 'Validation'])
plt.show()

# 8. Plot loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.legend(['Training', 'Validation'])
plt.show()
